# Baseline: ResNet-18 on the face crop

Runs `checkpoint1.baseline` (seed 228). Run `checkpoint1.preprocess` first. Settings: `CELEBA_SPOOF_ROOT`, `BASELINE_N_TRAIN` / `BASELINE_N_VAL` / `BASELINE_N_TEST` (N or `all`), `BASELINE_EPOCHS`, `BASELINE_WORKERS`, `BASELINE_RECOMPUTE=1` to retrain.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

if Path.cwd().name == 'notebooks':
    os.chdir('..')

from sklearn.metrics import roc_auc_score

from checkpoint1 import baseline, data

manifest_dir = Path(os.environ.get('BASELINE_MANIFEST_DIR', 'data/processed'))
out_dir = Path(os.environ.get('BASELINE_OUT_DIR', 'outputs/baseline'))
sizes = {split: baseline.parse_size(os.environ.get(f'BASELINE_N_{split.upper()}', str(baseline.DEFAULT_SIZES[split]))) for split in baseline.SPLITS}
epochs = int(os.environ.get('BASELINE_EPOCHS', 3))
workers = int(os.environ['BASELINE_WORKERS']) if 'BASELINE_WORKERS' in os.environ else None
recompute = os.environ.get('BASELINE_RECOMPUTE') == '1'

history, scores = baseline.run_baseline(None, manifest_dir, out_dir, sizes, epochs=epochs, workers=workers,
                                        recompute=recompute)
test = scores['test']


def show(name):
    display(Image(filename=out_dir / 'figures' / f'{name}.png', width=700))

## Training

The epoch with the best validation AUC is kept.

In [ ]:
history

## Performance metrics (test)

FAR = attacks accepted as live, FRR = live faces rejected, F1 with spoof as the positive class.

In [ ]:
test_eer, eer_threshold = baseline.eer(test['is_spoof'], test['score'])
print(f"test AUC: {roc_auc_score(test['is_spoof'], test['score']):.4f}")
print(f"test EER: {test_eer:.2%} at threshold {eer_threshold:.4f}")
pd.DataFrame([baseline.metrics(test, 0.5), baseline.metrics(test, eer_threshold), baseline.metrics(test, 0.1)])

## Threshold analysis

In [ ]:
show('base_far_frr')

## Error analysis

Error rate per presentation type at threshold 0.5.

In [ ]:
show('base_per_attack')

Error rate by capture condition.

In [ ]:
display(baseline.per_group(test, 'illumination', data.ILLUMINATION))
display(baseline.per_group(test, 'environment', data.ENVIRONMENT))